# Customize Tuner for Neural Network Optimization

In this notebook, we explore how to customize hyperparameter optimization for a model:
- Use different hyperparameter selections or settings for the standard models
- Optimize only part of a pipeline (e.g. tune only the classifier while keeping the embedder fixed) using `PipelineWithHyperparameterRooting` and a custom `AbstractMotherTuner`
- Define a custom model and use it in hyperparameter optimization

## Key concepts from `mother.optimization.core`

### `AbstractMotherTuner`
An abstract base class for building custom tuners. Subclasses must implement two methods:
- **`objective(trial, context)`** — defines a single Optuna trial: samples hyperparameters, trains and evaluates the model, and returns a scalar score.
- **`call_optimize(context)`** — controls the overall optimization loop, i.e. how `study.optimize()` is called with `self.objective`.

The `optimize()` method (inherited from `AbstractMotherTuner`) orchestrates the full workflow: it creates the Optuna study, assembles the `ObjectiveContext`, enqueues any default parameters, and finally retrains the best model on the full dataset.

### `ObjectiveContext`
A dataclass that bundles all arguments from `optimize()` into a single object passed to `objective()` and `call_optimize()`. Its key fields are:

| Field | Description |
|---|---|
| `get_hyper_space` | Callable that, given a trial, returns a hyperparameter dict |
| `estimator` | The unfitted pipeline to be tuned |
| `X` / `y` | Training data and targets |
| `cross_validation` | Cross-validator (e.g. `KFold`) |
| `groups` | Optional group labels for grouped CV |
| `fit_kwargs` | Extra keyword arguments forwarded to `estimator.fit()` |
| `extras` | Arbitrary additional data your custom tuner may need |

In [12]:
from collections import OrderedDict
from optuna import Trial

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import accuracy_score
from sklearn.base import BaseEstimator, clone, check_is_fitted
from sklearn.exceptions import NotFittedError

from mother.ml.models.m_tabpfn import TabPFNEmbeddingTransformer
from mother.ml.models.m_catboost import CatboostClassifierMother
from mother.ml import PipelineWithHyperparameterRooting
import mother.optimization as opt
from mother.ml import AbstractMotherPipeline
from mother.ml.models.utils import add_prefix_to_dict_keys

from torch.optim import Adam
from torch.utils.data import TensorDataset, DataLoader
from torch import nn
import torch

import pandas as pd


# Load a binary classification example dataset as pandas objects.
X, y = load_breast_cancer(return_X_y=True, as_frame=True)
# Reserve a holdout set for the standalone model demo. (the tuner below creates its own validation folds)
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2)
print(
    f"Train / Validation samples = {X_train.shape[0]} / {X_valid.shape[0]} with",
    f"{X_train.shape[1]} features and {y_train.nunique()} labels.",
)

Train / Validation samples = 455 / 114 with 30 features and 2 labels.


## 1. Optimize Part of a Pipeline

With a custom optimizer, you can optimize only a part of your pipeline.

Here, we optimize the classification model in a pipeline consisting of `TabPFNEmbeddingTransformer` (to generate model-based embeddings from the input features) and `CatboostClassifier`.

### 1.1 Create a Pipeline

In [ ]:
# Keep the embedder fixed and let the custom tuner search classifier parameters.
model = PipelineWithHyperparameterRooting(
    [
        (
            "embedder",
            TabPFNEmbeddingTransformer(
                model_type="classification",
                use_kfold=False,  # these parameters will not change after optimization
            ),
        ),
        ("classifier", CatboostClassifierMother(target_type="single_target", logging_level="Silent")),
    ]
)

model.steps

[('embedder', TabPFNEmbeddingTransformer(use_kfold=False)),
 ('classifier',
  CatboostClassifierMother(auto_class_weights='Balanced', boosting_type='Plain', bootstrap_type='MVS', grow_policy='SymmetricTree', learning_rate=0.03, logging_level='Silent', loss_function='Logloss', max_depth=6, posterior_sampling=True, random_strength=1))]

In an sklearn pipeline, parameter names indicate which step they belong to using the convention `{step_name}__{parameter_name}`.

In [3]:
# Inspect nested sklearn parameter names, such as classifier__... and embedder__....
model.get_params()

{'memory': None,
 'steps': [('embedder', TabPFNEmbeddingTransformer(use_kfold=False)),
  ('classifier',
   CatboostClassifierMother(auto_class_weights='Balanced', boosting_type='Plain', bootstrap_type='MVS', grow_policy='SymmetricTree', learning_rate=0.03, logging_level='Silent', loss_function='Logloss', max_depth=6, posterior_sampling=True, random_strength=1))],
 'transform_input': None,
 'verbose': False,
 'embedder': TabPFNEmbeddingTransformer(use_kfold=False),
 'classifier': CatboostClassifierMother(auto_class_weights='Balanced', boosting_type='Plain', bootstrap_type='MVS', grow_policy='SymmetricTree', learning_rate=0.03, logging_level='Silent', loss_function='Logloss', max_depth=6, posterior_sampling=True, random_strength=1),
 'embedder__device': 'cpu',
 'embedder__embedding_column_name': 'tabpfnembedding',
 'embedder__ignore_pretraining_limits': True,
 'embedder__model': None,
 'embedder__model_type': 'classification',
 'embedder__n_folds': 5,
 'embedder__random_state': None,
 'e

### 1.2 Custom Optimizer

In the custom `objective` function, we select only parameters whose names start with `classifier`, so that only the classifier is optimized while the embedder remains fixed.
`optimize()` is inherited from `opt.AbstractMotherTuner`. It prepares the study and `ObjectiveContext`, then calls `self.call_optimize(context)`.
Once trials finish, the inherited method applies the best parameters and refits a clone on all supplied data.

In [ ]:
class CustomMotherTuner(opt.AbstractMotherTuner):
    def __init__(self, **kwargs):
        # Keep study configuration and callbacks from the base tuner.
        super().__init__(**kwargs)

    def objective(self, trial: Trial, context: opt.ObjectiveContext) -> float:
        cv_score = 0.0

        # Ask the pipeline for all suggested parameters, then restrict tuning to the "classifier" step.
        suggested_params_to_train: dict = context.get_hyper_space(trial=trial, X=context.X, y=context.y)
        suggested_params_to_train = {k: v for k, v in suggested_params_to_train.items() if k.startswith("classifier")}

        print(f"Trial {trial.number} parameters to tune : {suggested_params_to_train}")

        # Evaluate the same trial settings across each cross-validation split.
        for train_idx, test_idx in context.cross_validation.split(context.X, context.y):
            # Clone per fold so fitted state is not carried into the next split.
            pipeline = clone(context.estimator)

            # fit
            pipeline.set_params(**suggested_params_to_train)
            pipeline.fit(context.X.iloc[train_idx, :], context.y.iloc[train_idx])

            # Accumulate held-out accuracy, then average over all folds.
            y_pred_test = pipeline.predict(X=context.X.iloc[test_idx, :])
            cv_score += accuracy_score(context.y.iloc[test_idx], y_pred_test)

        return cv_score / (context.cross_validation.get_n_splits())  # mean accuracy

    def call_optimize(self, context: opt.ObjectiveContext) -> None:
        # Trial scheduling to Optuna and run optimization
        self.study.optimize(
            lambda trial: self.objective(trial, context=context),
            n_trials=self.n_trials_optuna,
            gc_after_trial=True,
            callbacks=self.get_callbacks(),
        )


# Maximize mean CV accuracy (the small trial count keeps this example quick).
tuner = CustomMotherTuner(
    n_trials_optuna=3,  # number of trials for hyperparameter optimization
    n_threads_optuna=10,  # parallel threads for cross-validation evaluation
    n_startup_trials=1,  # number of random trials before using optuna
    tuning_direction="maximize",  # Maximize the accuracy
)


# Use a small subset here to keep the demonstration runtime manageable.
model_tuned = tuner.optimize(
    model,
    X_train.iloc[:100, :],
    y_train.iloc[:100],
    cross_validation=KFold(n_splits=2),
    hyperparameter_space_function=model.get_hyperparameter_space,
)

/workspaces/MotherML/src/mother/optimization/core.py:142: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  self.sampler = optuna.samplers.TPESampler(
/workspaces/MotherML/src/mother/optimization/core.py:142: ExperimentalWarning: Argument ``group`` is an experimental feature. The interface can change in the future.
  self.sampler = optuna.samplers.TPESampler(
/workspaces/MotherML/src/mother/optimization/core.py:142: ExperimentalWarning: Argument ``constant_liar`` is an experimental feature. The interface can change in the future.
  self.sampler = optuna.samplers.TPESampler(


Trial 0 parameters to tune : {'classifier__bootstrap_type': 'MVS', 'classifier__learning_rate': 0.03, 'classifier__random_strength': 1, 'classifier__grow_policy': 'SymmetricTree', 'classifier__max_depth': 6, 'classifier__loss_function': 'Logloss'}


Trial 1 parameters to tune : {'classifier__bootstrap_type': 'MVS', 'classifier__learning_rate': 0.008962457743507004, 'classifier__random_strength': 0.6761980470225799, 'classifier__grow_policy': 'SymmetricTree', 'classifier__max_depth': 7, 'classifier__loss_function': 'Logloss'}
Trial 2 parameters to tune : {'classifier__bootstrap_type': 'MVS', 'classifier__learning_rate': 0.038418103667855155, 'classifier__random_strength': 1.2928874937660844, 'classifier__grow_policy': 'Lossguide', 'classifier__boosting_type': 'Plain', 'classifier__max_depth': 7, 'classifier__max_leaves': 26, 'classifier__loss_function': 'Logloss'}
{'classifier__bootstrap_type': 'MVS', 'classifier__learning_rate': 0.03, 'classifier__random_strength': 1.0, 'classifier__grow_policy': 'SymmetricTree', 'classifier__max_depth': 6, 'classifier__loss_function': 'Logloss'}


After optimization, we can verify that the embedder parameters remain unchanged.

In [5]:
# Confirm the embedder's fixed setting was preserved in the optimized pipeline.
model_tuned.get_params()["embedder__use_kfold"]

False

## 2. Mother Optimization for a PyTorch Model

In this example, we wrap a PyTorch neural network in `AbstractMotherPipeline` and tune it with a custom optimizer. The following Torch APIs handle model construction and training; they are separate from Mother's tuner interface:

- **`nn.Sequential`** defines the network (two linear layers with a ReLU between them and a sigmoid output for binary-class probabilities in the code below).
- **`TensorDataset` and `DataLoader`** pair feature tensors with labels and provide batches for training and validation. Shuffling is enabled for training batches.
- **`BCELoss` and `Adam`** define the binary-classification loss and the optimizer that updates network weights.
- **`zero_grad()`, `backward()`, and `step()`** perform one training update: clear old gradients, calculate new gradients, then update the weights.
- **`train()`, `eval()`, and `torch.no_grad()`** switch between training and evaluation behavior and avoid recording gradients during validation.

The wrapper's `fit`, `default_parameters`, and `get_hyperparameter_space` methods connect this PyTorch training loop to Mother's estimator and tuning APIs. For a broader introduction, see the official [PyTorch Learn the Basics tutorial](https://pytorch.org/tutorials/beginner/basics/intro.html).

### 2.1 Torch Wrapper for Mother
In this example, we only tune the learning rate, but additional parameters can be added for optimization.

In [ ]:
# This wrapper provides the estimator and hyperparameter hooks expected by Mother.
class TorchNNMother(AbstractMotherPipeline, BaseEstimator):
    def __init__(self, lr: float = 1e-3, **kwargs) -> None:
        # Produce a class probability of the 30-feature breast cancer dataset.
        self.network = nn.Sequential(
            OrderedDict(
                [
                    ("linear1", nn.Linear(in_features=30, out_features=10)),
                    ("relu", nn.ReLU()),
                    ("linear2", nn.Linear(in_features=10, out_features=1)),
                    ("sigmoid", nn.Sigmoid()),
                ]
            )
        )

        self.optimizer = Adam(self.network.parameters(), lr=lr)
        self.lr = lr
        self.loss = nn.BCELoss()
        # Retain constructor parameters so sklearn cloning and tuning can rebuild this estimator.
        self._init_params: dict = {"lr": lr}

        non_optimised_params: list[str] = ["_init_params"]
        for k, v in kwargs.items():
            if k not in non_optimised_params:
                self._init_params[k] = v

    def default_parameters(self, prefix: str = "") -> dict:
        # Supply the default learning rate using the requested pipeline prefix.
        return add_prefix_to_dict_keys({"lr": 1e-3}, prefix=prefix)

    def get_hyperparameter_space(self, X, y, trial: Trial, prefix: str = "") -> dict:
        # hyper parameter search for learning rate
        suggested_params: dict = {"lr": trial.suggest_float(prefix + "lr", 1e-5, 1e-3, log=True)}
        suggested_params = add_prefix_to_dict_keys(suggested_params, prefix=prefix)

        return suggested_params

    def get_params(self, deep=True) -> dict:
        # Expose initialization parameters to sklearn's estimator utilities.
        return self._init_params

    def set_params(self, **params):
        for key, value in params.items():
            if key in self._init_params.keys():
                self._init_params[key] = value

        # Rebuild runtime state after updating parameters so the optimizer uses the tuned rate.
        self.lr = float(self._init_params["lr"])

        return self.__init__(**self._init_params)

    def _get_dataloader(self, X, y, batch_size: int = 128, shuffle: bool = False) -> DataLoader:
        # Convert pandas inputs to tensors and package features and labels for PyTorch.
        if isinstance(X, pd.DataFrame):
            X = X.values
        if isinstance(y, pd.Series):
            y = y.values
        return DataLoader(
            TensorDataset(
                torch.tensor(X).to(torch.float32),
                torch.tensor(y).reshape(-1, 1).to(torch.float32),
            ),
            batch_size=batch_size,
            shuffle=shuffle,
        )

    def validation(self, X_valid, y_valid):
        # Disable gradient tracking while measuring held-out loss and thresholded accuracy.
        valid_data_loader = self._get_dataloader(X_valid, y_valid)

        self.network.eval()
        valid_loss = 0.0
        valid_acc = 0.0

        with torch.no_grad():
            for X, y in valid_data_loader:
                y_hat = self.network(X)
                valid_loss += self.loss(y_hat, y).item() / len(valid_data_loader)
                y_hat_pred = (y_hat > 0.5).float()
                valid_acc += accuracy_score(
                    y.detach().numpy().ravel(),
                    y_hat_pred.detach().numpy().ravel(),
                ) / len(valid_data_loader)

        return valid_loss, valid_acc

    def fit(self, X_train, y_train, n_epochs: int = 100):
        train_data_loader = self._get_dataloader(X_train, y_train, shuffle=True)

        # Start each fit with a fresh optimizer configured for the current trial's learning rate.
        self.optimizer = Adam(self.network.parameters(), lr=float(self._init_params["lr"]))

        for epoch in range(n_epochs):
            self.network.train()
            train_loss = 0.0
            for X, y in train_data_loader:
                self.optimizer.zero_grad()
                y_hat = self.network(X)
                batch_train_loss = self.loss(y_hat, y)

                if not torch.isfinite(batch_train_loss):
                    raise ValueError("Training diverged (non-finite loss). Try a lower learning rate.")

                batch_train_loss.backward()
                self.optimizer.step()
                train_loss += batch_train_loss.item() / len(train_data_loader)
        return self


# Run the wrapper once with its default learning rate and inspect validation metrics.
model = TorchNNMother(lr=1e-3)
model.fit(X_train, y_train)
print(model.validation(X_valid, y_valid))

(42.98245620727539, 0.5701754385964912)


The cell below checks sklearn estimator inheritance, cloning support, and fitted-state detection for `TorchNNMother`.

In [ ]:
print(f"TorchNNMother subclasses sklearn BaseEstimator: {issubclass(TorchNNMother, BaseEstimator)}")
print(f"model is an sklearn BaseEstimator: {isinstance(model, BaseEstimator)}")
cloned_model = clone(model)
print(f"sklearn clone returns TorchNNMother: {isinstance(cloned_model, TorchNNMother)}")
try:
    check_is_fitted(model)
except NotFittedError:
    print("Model is not fitted yet.")

cloned_model

TorchNNMother subclasses sklearn BaseEstimator: True
model is an sklearn BaseEstimator: True
sklearn clone returns TorchNNMother: True
Model is not fitted yet.


,lr,0.001


### 2.2 Custom Mother Tuner for Torch

Neural network training often requires splitting data into three sets (train, validation, and test). This does not align with the default `MotherTuner` design, which does not create a validation set for monitoring training epochs. To address this, we add a train-validation split inside the custom objective function.

In [ ]:
# This objective creates a validation split inside each trial because no CV splitter is used.
class TorchMotherTuner(opt.AbstractMotherTuner):
    def __init__(self, **kwargs):
        # Reuse the base tuner's study setup and configuration.
        super().__init__(**kwargs)

    def objective(self, trial: Trial, context: opt.ObjectiveContext) -> float:
        # Use a held-out portion of this trial's data to compare training runs.
        X_train, X_valid, y_train, y_valid = train_test_split(context.X, context.y, test_size=0.2)
        print(
            f"Train / Validation samples = {X_train.shape[0]} / {X_valid.shape[0]} with {X_train.shape[1]} features",
            f" and {y_train.nunique()} labels.",
        )

        # Clone the estimator so trial-specific parameter updates stay isolated.
        estimator = clone(context.estimator)
        suggested_params_to_train: dict = context.get_hyper_space(trial=trial, X=context.X, y=context.y)
        estimator.set_params(**suggested_params_to_train)
        estimator.fit(X_train, y_train)

        # Return validation loss because this study is configured to minimize.
        valid_loss, valid_acc = context.estimator.validation(X_valid, y_valid)

        return valid_loss

    def call_optimize(self, context: opt.ObjectiveContext) -> None:
        # Run the custom objective through Optuna with the shared callbacks.
        self.study.optimize(
            lambda trial: self.objective(trial, context=context),
            n_trials=self.n_trials_optuna,
            gc_after_trial=True,
            callbacks=self.get_callbacks(),
        )


# Minimize validation loss for the learning-rate search.
tuner = TorchMotherTuner(
    n_trials_optuna=3,  # number of trials for hyperparameter optimization
    n_threads_optuna=10,  # parallel threads for cross-validation evaluation
    n_startup_trials=1,  # number of random trials before using optuna
    tuning_direction="minimize",  # Need to minimize the loss!
)

# Pass the full dataset; each objective call makes its own train-validation split.
model_tuned = tuner.optimize(
    model,
    X,
    y,
    cross_validation=None,
    hyperparameter_space_function=model.get_hyperparameter_space,
)

Train / Validation samples = 455 / 114 with 30 features and 2 labels.


/workspaces/MotherML/src/mother/optimization/core.py:142: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  self.sampler = optuna.samplers.TPESampler(
/workspaces/MotherML/src/mother/optimization/core.py:142: ExperimentalWarning: Argument ``group`` is an experimental feature. The interface can change in the future.
  self.sampler = optuna.samplers.TPESampler(
/workspaces/MotherML/src/mother/optimization/core.py:142: ExperimentalWarning: Argument ``constant_liar`` is an experimental feature. The interface can change in the future.
  self.sampler = optuna.samplers.TPESampler(


Train / Validation samples = 455 / 114 with 30 features and 2 labels.
Train / Validation samples = 455 / 114 with 30 features and 2 labels.
{'lr': 0.0009361658778024464}
